# SVG Inference — Load Merged Model & Generate Submission

Inference-only notebook. Loads the pre-trained merged model from Kaggle kernel output and generates `submission.csv` for the 1,000 test prompts.

**Before running:** Add the training kernel output as a data source in Kaggle (Add Data → Kernel Outputs → search `ryanmfleishman/dl-llama-version`). The merged model will then be at `/kaggle/input/dl-llama-version/qwen25_coder_svg_lora_merged`.

In [ ]:
%pip install -q unsloth transformers accelerate peft bitsandbytes pandas lxml

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths
MERGED_MODEL_DIR = "/kaggle/input/dl-llama-version/qwen25_coder_svg_lora_merged"
TEST_PATH        = "/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"
MAX_NEW_TOKENS   = 256

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model path exists: {os.path.isdir(MERGED_MODEL_DIR)}")

In [ ]:
# SVG validation helpers
MAX_SVG_CHARS = 16_000
MAX_PATH_ELEMENTS = 256
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter",
}


def _local_tag(element):
    tag = element.tag
    return tag.split("}", 1)[-1] if "}" in tag else tag


def is_valid_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return False
    if len(svg_text) > MAX_SVG_CHARS:
        return False
    svg_text = svg_text.strip()
    if not svg_text.lower().startswith("<svg"):
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if _local_tag(root) != "svg":
        return False
    path_count = 0
    for el in root.iter():
        tag = _local_tag(el)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_ELEMENTS:
        return False
    return True


def fallback_svg(_prompt=""):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


assert is_valid_svg(fallback_svg()), "Fallback SVG must be valid"
print("SVG validation helpers ready.")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MERGED_MODEL_DIR,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None
print("Model loaded.")

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from a natural language description. "
    "Output only SVG code. The root element must be <svg> with width=\"256\" height=\"256\" "
    "viewBox=\"0 0 256 256\". Use only allowed tags. Keep the output under 16000 characters."
)

SVG_REGEX = re.compile(r"<svg.*?</svg>", flags=re.IGNORECASE | re.DOTALL)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def generate_svg(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )
    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    svg = extract_svg(decoded)
    return svg if is_valid_svg(svg) else fallback_svg()


# Smoke test
test_svg = generate_svg("a simple blue circle on a white background")
print(f"Smoke test — valid: {is_valid_svg(test_svg)}, length: {len(test_svg)}")
print(test_svg[:200])

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test rows: {len(test_df)}")

all_ids     = test_df["id"].tolist()
all_prompts = test_df["prompt"].tolist()

rows = []
invalid_count = 0
t0 = time.time()

for i, (id_, prompt) in enumerate(zip(all_ids, all_prompts)):
    svg = generate_svg(prompt)
    if not is_valid_svg(svg):
        invalid_count += 1
    rows.append({"id": id_, "svg": svg})
    if (i + 1) % 10 == 0 or (i + 1) == len(all_prompts):
        elapsed = (time.time() - t0) / 60
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        eta = (len(all_prompts) - i - 1) / rate if rate > 0 else 0
        print(f"  {i+1}/{len(all_prompts)} | {elapsed:.1f} min elapsed | ETA {eta:.1f} min | {invalid_count} fallbacks")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\nSaved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Fallbacks: {invalid_count} ({100*invalid_count/len(sub_df):.1f}%)")
print(f"Runtime: {elapsed_min:.1f} min")
sub_df.head()